# Normalization Template
The purpose of this notebook is to provide a template for normalization across all of the datasets.  
I will be running this on the Franklin Dataset as an example.

Load Libraries

In [32]:
import pandas as pd
import glob
import os
import re

## Reason Column Normalization
The following cells describe some global normalization mappings we will use one the reasons category. There is also a mapping to make the months more readable.  

**NOTE**: The exact mappings were generated by claude AI to account for all reasons presented in each dataset. If there is any issue in the reasoning behind some of the values mapping, contact the group so that everyones dataset are normalized on the same values.

In [33]:
MONTHS = {
    "jan": "01", "feb": "02", "mar": "03", "apr": "04",
    "may": "05", "jun": "06", "jul": "07", "aug": "08",
    "sep": "09", "oct": "10", "nov": "11", "dec": "12",
}

CATEGORY_KEYWORDS = [
    ("Missing / Endangered Person", [
        # "missing plate/tag/vehicle" are NOT missing persons (handled by Traffic / Stolen Vehicle)
        r"missing(?!\s+(?:license\s+|front\s+|rear\s+)?(?:plate|tag)s?\b)(?!\s+(?:veh\w*|car|truck|motorcycle)\b)",
        r"endangered", r"\bamber\b", r"silver alert", r"runaway", r"we?ll?fare", r"\bsuic\w*",
        r"critical (?:missing|person)", r"\bawol\b", r"lost (?:child|person)", r"vulnerable adult", r"\bmental\b"]),
 
    ("Violent / Sexual Crime", [
        r"shooting", r"shootin\b", r"shots?\s*fired", r"\bshot\b", r"hom[io]?cid\w*", r"murder", r"\brob+e?ry\b", r"\brobery\b",
        r"carjack", r"car\s*jack", r"kidnap", r"abduc", r"assault", r"\braven\b", r"\bagg\b", r"aggravated", r"battery",
        r"stalk", r"harass", r"threat", r"domestic", r"\brape\b", r"\bsex\w*", r"molest", r"child abuse", r"icac", r"\bcsam\b",
        r"human traffick\w*", r"weapon", r"firearm", r"\bguns?\b", r"stabb?ing", r"\bstab\b", r"death investigation",
        r"\bhostage", r"strangul", r"\barmed\b", r"extortion", r"intimidat", r"\bgang\b", r"\bbomb\b", r"\bterror\w*",
        r"manslaughter", r"\bviolan?ce\b", r"\bviolen\w*", r"child (?:porn|exploit)\w*", r"home invasion",
        r"custodial interference", r"protection order", r"(?<!vin )\bswitch(?:es)?\b", r"\bdraco\b", r"bullet hole",
        r"tren de aragua", r"\bagv\b", r"\bass\b"]),
 
    ("Stolen Vehicle / MVT", [
        r"vin\s*(?:swap|switch|clone|cloning)", r"\bre[-\s]?vin\b", r"missing\s+(?:veh\w*|car|truck|motorcycle)\b",
        r"stolen", r"\bsto;?len\b", r"\bstol\b", r"\bmvt\b", r"\bstole\w*", r"motor vehicle theft",
        r"(?:vehicle|veh|auto|car|truck|plate|tag|motorcycle|trailer)\s+theft",
        r"theft\s+(?:of\s+)?(?:a\s+)?(?:vehicle|veh|auto|car|truck|plate|tag)",
        r"suspected theft veh\w*", r"\bgta\b", r"grand theft auto"]),
 
    ("Warrant / Wanted Person", [
        r"warr?ant\w*", r"\bwanted\b", r"fugitive", r"\bwar\b", r"capias", r"indictment", r"\bbench\b", r"probation", r"parole",
        r"abscond", r"\bfta\b", r"contempt", r"apprehen", r"manhunt", r"escap(?:e|ee)\b", r"\barrest\w*", r"\bwtd\b",
        r"\bvops?\b", r"extradition",
        r"\b10-?17s?\b"]),   # local code: "10-17(s)" appears alongside warrants/VOP in the Knoxville data
 
    ("BOLO / Suspect Vehicle", [
        r"\bbolo\b", r"\batl\b", r"attempt to locate", r"suspect", r"suspicious", r"vehicle of interest", r"\bsus\b", r"\bsusps?\b",
        r"\bsuspic\w*", r"\bpoi\b", r"person of interest", r"\bvoi\b", r"lookout",
        r"\bfled\b", r"\bfle(?:e|eing|ein)\w*", r"\bflee\w*", r"evad\w*", r"elud\w*", r"pursuit",   # fleeing / evading
        r"\bbulletin\b", r"hot\s*sheet"]),
 
    ("Drug Investigation", [
        r"drug", r"narcotic", r"overdose", r"\bod\b", r"\bmeth\w*", r"fentanyl", r"cocaine", r"heroin", r"marijuana",
        r"cannabis", r"sale delivery", r"\bnarc\w*", r"\bcds\b", r"\bpwid\b"]),
 
    ("Property Crime", [
        r"bu?r?glar\w*", r"burg", r"theft", r"\bdamage\b", r"larceny", r"fraud", r"shoplift", r"vandal", r"destruction",
        r"trespass", r"arson", r"embezzl", r"forger", r"counterfeit", r"identity", r"credit card", r"financial",
        r"\bproperty\b", r"found prop", r"break(?:ing)?\s*(?:and|&|in)", r"\bb\s*&\s*e\b", r"\bscam", r"catalytic", r"\bthief",
        r"skimm\w*", r"pickpocket", r"\borc\b", r"\broc\b", r"rocic", r"quick change", r"organized retail"]),
 
    ("Traffic / Crash", [
        r"traffic", r"infraction", r"hit\s*(?:and|&|n)?\s*run", r"\bh\s*&\s*r\b", r"\bh/r\b", r"crash", r"accident",
        r"\bmvc\b", r"\bmva\b", r"\bdui\b", r"\bdwi\b", r"impaired", r"intoxicat", r"\bspeed\w*", r"reck\w*ss?", r"road rage",
        r"motor vehicle offense", r"citation", r"tailgat", r"expired", r"uninsured", r"\binsurance\b", r"registration",
        r"revoked", r"suspended", r"missing (?:license )?(?:plate|tag)", r"tag abuse", r"fake (?:temp )?tags?", r"road hazard",
        r"temp(?:orary)? tags?", r"dealer (?:tag|plate)", r"\bnof\b", r"not on file", r"mex(?:ican)? plate",
        r"covered (?:license )?plate", r"tag displayed", r"street rac\w*", r"side\s*show", r"\babandoned\b"]),
 
    # Lower-priority local shorthand / hotlist notes. Only reached when NO offense word matched above,
    # so "Drugs - own PC" stays Drug but a bare "Own PC - notes" becomes BOLO / Suspect Vehicle.
    #   "44 ..."  (Murfreesboro): always a vehicle alert ("44 Veh Bolo", "44 tag alert", "44 Veh Hotlist Alert")
    #   "53s"     (Murfreesboro): almost always follows "ATL" ("ATL - 53s")
    #   "own/develop PC" + "notify ... to remove" (Knoxville): hotlist notes on vehicles of interest
    ("BOLO / Suspect Vehicle", [
        r"\b44\b", r"\b44\s*(?:veh\w*|tag|plate|alert|bolo|s)\b", r"\b53['’]?s?\b", r"\b(?:own|develop|independent)\s+pc\b", r"do not stop", r"remov(?:al|e)\b",
        r"\bran from\b", r"took off", r"potential veh", r"\bflg\b", r"hotlist", r"veh photo"]),
 
    ("Routine / Administrative / Test", [
        r"patrol", r"\bpat\b", r"\bpatr\b", r"\bpatro\b", r"routine", r"\btest\w*", r"training", r"image download",
        r"daytime search", r"more like this", r"information only", r"info only", r"\binfo\b", r"\binformation\b", r"\bdemo\b",
        r"\baudit\b", r"maintenance", r"carfax", r"high mileage", r"\bdownload\b", r"image search", r"\blpr\b",
        r"new install", r"media interview", r"angle review"]),
 
    ("General Investigation", [
        r"\binv(?:g|gt|\.|\b)", r"\binve\b", r"\binve?s?t\w*", r"\binvetigat\w*", r"\bines?tig\w*", r"\bcase\b", r"investigat",
        r"\bsid\b", r"\btbi\b", r"law enforcement", r"\ble\b", r"task\s*force", r"interdiction", r"\bintel\w*", r"criminal",
        r"felony", r"misdemeanor", r"\bcomplaint\b", r"\bdet\b\.?", r"\bdet\w+\b", r"\bdetective\b", r"\bsgt\b", r"\bdea\b",
        r"\bfbi\w*", r"\batf\b", r"\bhsi\b", r"\bnic\b", r"\bncic\b", r"\bsearch\b", r"\bactive incident\b", r"\bincident\b",
        r"\brcso\b", r"\bkpd\b", r"\bcrime\b", r"\boffense\b", r"witness", r"\bfelon\b", r"\bvictim", r"follow\s*-?\s*up",
        r"\bfu\b", r"\bassist\w*", r"\brequest\b", r"attempt to id", r"\bid needed\b", r"needs? occp", r"\bcops\b",
        r"\bco so\b", r"\bso request\b", r"\bpd\b", r"sheriff", r"\buc\b", r"utpdcase", r"\bsvu\b", r"\bsfu\b", r"\bfact\b",
        r"officer safety", r"\bdeal\b"]),
]

_COMPILED = [(cat, re.compile("|".join(f"(?:{p})" for p in pats), re.I)) for cat, pats in CATEGORY_KEYWORDS]

# --- pre-processing helpers -------------------------------------------------------------------------
_ALERT_PREFIX = re.compile(r"other image search from alerts page associated with alert:?\s*", re.I)      # system-generated
_ALERT_TS = re.compile(r"\s*-?\s*\d{4}-\d{2}-\d{2}t[\d:.]+z\s*$", re.I)
_ALERT_PLATE = re.compile(r"\s+(?=[a-z0-9]*\d)[a-z0-9]{6,8}\s*$", re.I)                                   # trailing plate token
_CASE_NO = re.compile(r"""(?ix)
    \bmppat[-\s]?\d{2}[-\s]?\d+\b | \bmp+a+t?[-\s]?\d{2}[-\s]?\d+\b | \b\d{2}[-\s]\d{3,}\b
  | \b[a-z]?\d{8,}[a-z]{0,2}\b | \b\d{5,}\b | \b\d{2}[a-z]{2}\d[-\s]?\d{4}[-\s]?[a-z]{2}[-\s]?\d+\b""")
_MONTHS = "jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec"
_DATE_JUNK = re.compile(rf"^(?:{_MONTHS})[-\s]?\d{{1,2}}$|^\d{{1,2}}[-\s](?:{_MONTHS})$|^\d{{1,2}}/\d{{1,2}}/\d{{2,4}}(?:\s+\d+:\d+)?$", re.I)
_SPREADSHEET_ERR = re.compile(r"^#(?:name|ref|value|n/a|div)|^(?:undefined|null|nan|n/?a)$", re.I)
_STOLEN_PROPERTY = re.compile(r"stolen property|receiving stolen", re.I)                                    # property, not vehicle
_MISC_WORDS = re.compile(r"\bvice\b|false report|lost vehicle|sov(?:ereign)? citizen|disorderly|loiter|riot|prostitut|curfew|massage|drive off", re.I)
_PLATE_LIKE = re.compile(r"^(?=.*\d)(?=.*[a-z])[a-z0-9]{5,8}$", re.I)

ID_ONLY = "No Offense Stated / Redacted"
MALFORMED = "Malformed / Unintelligible"
MISC = "Misc / Other"


Normalization helper functions

In [34]:
def fix_corrupted_code(text):
    if text.endswith("00:00:00"):
        date_part = text.split(" ")[0]
        pieces = date_part.split("-")
        if len(pieces) == 3:
            y, m, d = pieces
            if y.startswith("19"):
                return m + "-" + y[-2:]
            else:
                return m + "-" + d
        return text

    parts = text.split("-")
    if len(parts) == 2:
        a, b = parts[0].strip(), parts[1].strip()
        if a in MONTHS:
            return MONTHS[a] + "-" + b
        if b in MONTHS:
            return MONTHS[b] + "-" + a

    return text


def normalize_reason(raw):
    if pd.isna(raw):
        return ""
    text = str(raw).strip().lower()
    if text == "":
        return ""

    text = fix_corrupted_code(text)

    text = text.replace("+", " ")
    text = text.replace("_", " ")
    text = " ".join(text.split())

    return text


def is_radio_code(text):
    stripped = text.replace("-", "")
    return stripped.isdigit()

def categorize_reason(raw):
    """Return the category name for one raw reason string."""
    s = re.sub(r"\s+", " ", (raw or "").replace("+", " ").replace("­", "")).strip()   # '+' = URL-encoded space

    if not s or _SPREADSHEET_ERR.match(s) or _DATE_JUNK.match(s):                            
        return MALFORMED
 
    is_alert = bool(_ALERT_PREFIX.search(s))
    text = _ALERT_TS.sub("", _ALERT_PREFIX.sub("", s))

    if is_alert:
        text = _ALERT_PLATE.sub("", text.strip())
    has_case_no = bool(_CASE_NO.search(text))
    text = _CASE_NO.sub(" ", text)         
 
    if _STOLEN_PROPERTY.search(text):
        return "Property Crime"
    for cat, rx in _COMPILED:
        if rx.search(text):
            return cat
 
    # nothing matched
    if is_alert:
        return "Routine / Administrative / Test"        # system alert with no usable text
    
    if _MISC_WORDS.search(text):
        return MISC
    
    if has_case_no or _PLATE_LIKE.match(s):
        return ID_ONLY
    
    if len(re.sub(r"[^A-Za-z]", "", text)) <= 2:        # "p", "mp", "90", "1055"
        return MALFORMED
    
    no_digits = re.sub(r"\d+", " ", text)
    words = re.findall(r"[A-Za-z][A-Za-z.'’-]*", no_digits)
    if not re.sub(r"[A-Za-z.'’\-,\s]", "", no_digits) and len(words) <= 4:
        return ID_ONLY                                   # officer / subject names, initials, badge numbers
    return MISC

Load dataset and perform normalization

In [35]:
input_dir = "./datasets"
output_dir = "./categorized_output"

os.makedirs(output_dir, exist_ok=True)

summary_rows = []

for path in sorted(glob.glob(os.path.join(input_dir, "*.csv"))):
    city = os.path.splitext(os.path.basename(path))[0]
    df = pd.read_csv(path, dtype=str)

    df["reason_normalized"] = df["reason"].apply(normalize_reason)
    df["reason_category"] = df["reason_normalized"].apply(categorize_reason)

    out_path = os.path.join(output_dir, city + "_categorized.csv")
    df.to_csv(out_path, index=False)

    counts = df["reason_category"].value_counts()
    for category, count in counts.items():
        summary_rows.append({"city": city, "category": category, "count": count})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(output_dir, "category_summary.csv"), index=False)

pivot = summary_df.pivot_table(index="category", columns="city", values="count", fill_value=0, aggfunc="sum")
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)
print(pivot)

city                             Franklin  Knoxville  Memphis  Murfreesboro  \
category                                                                      
Malformed / Unintelligible           1316      33805    11043          2623   
No Offense Stated / Redacted          107       3306     1857         29166   
General Investigation                2689      12402    11805          3002   
BOLO / Suspect Vehicle                339       7385     2081          4382   
Stolen Vehicle / MVT                  655       1575     5926          1426   
Warrant / Wanted Person               121       5930     1813           941   
Violent / Sexual Crime                237       2131     2081          2643   
Routine / Administrative / Test       482        824      688           823   
Traffic / Crash                        39        481      140           630   
Drug Investigation                      0       2451       11           119   
Property Crime                        352        917

## Quick Categorical Analysis
### Total Rows: 177,997
### % Malformed Or Redacted: ~50%
### % Actual Offenses:       ~50%

The percentages above were found in order to ensure that we still had a large amount of rows in a offenses category. The actual split will depend on each dataset but it appears that we will have enough data to perform a meaningful analysis. 

**Note**: Murfreesboro has a extremely high amount of redacted searches. It may be worth looking into why the city has such a large amount of these compared to the other cities.

